In [ ]:
"""
ST422 Brief 8 — Road Safety Analysis
Combined Analysis Notebook
Purpose: Loads STATS19 data, cleans and merges, performs EDA, trend analysis,
         hotspot identification, contributory factor analysis, robustness checks,
         uncertainty quantification, and OLS + Poisson/NB diagnostics.

Inputs:
  Data/   — raw DfT STATS19 CSVs (see README.md for filenames)

Outputs:
  outputs/fig_*.png — all figures saved for traceability
  outputs/tab_*.csv — all key tables saved for traceability

Run order: Run this notebook top to bottom.
Restart kernel before a full re-run to guarantee a clean state.
"""

# ── 0. Config and Setup ───────────────────────────────────────────────────────

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

try:
    import pyarrow.csv as pa_csv
    import pyarrow.compute as pc
    HAS_ARROW = True
except ImportError:
    HAS_ARROW = False

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.family': 'sans-serif', 'figure.dpi': 120
})

NOTEBOOK_DIR = os.getcwd()
DATA_DIR     = os.path.join(NOTEBOOK_DIR, 'Data')
CLEANED_DIR  = os.path.join(NOTEBOOK_DIR, 'Cleaned')
OUTPUT_DIR   = os.path.join(NOTEBOOK_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

YEAR_START   = 2014
YEAR_END     = 2024
COVID_YEARS  = [2020, 2021]
BASE_YEARS   = [2015, 2016, 2017]
RECENT_YEARS = [2022, 2023, 2024]

MIN_KSI_BASE   = 30     # minimum mean annual KSI for LA to be included in OLS
MIN_YEARS      = 6      # minimum years of data required for OLS fit
LOW_R2         = 0.5    # R² threshold distinguishing linear from non-linear trends
TOP_N          = 5      # number of priority LAs to report
TOP_N_LINES    = 3      # number of LAs to show on trend line charts
HOTSPOT_TOP_N  = 15     # number of LAs in hotspot overlap check
MIN_COLLISIONS = 500    # minimum total collisions for LA to be included in rate ranking
THRESHOLD_PCT  = 15     # material change threshold (% above baseline)
N_BOOTSTRAP    = 1000   # bootstrap resamples for CI validation

# Policy evaluation — update to reflect the authorities under review
POLICY_LAS = ['Liverpool', 'Southwark']

# Choropleth boundary file URL
CHOROPLETH_URL = (
    'https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/'
    'Local_Authority_Districts_December_2024_Boundaries_UK_BGC/FeatureServer/0/query'
    '?where=1%3D1&outFields=LAD24CD,LAD24NM&outSR=4326&f=geojson'
)

COLLISION_HIST = os.path.join(DATA_DIR, 'dft-road-casualty-statistics-collision-1979-latest-published-year.csv')
CASUALTY_HIST  = os.path.join(DATA_DIR, 'dft-road-casualty-statistics-casualty-1979-latest-published-year.csv')
VEHICLE_HIST   = os.path.join(DATA_DIR, 'dft-road-casualty-statistics-vehicle-1979-latest-published-year.csv')
COLLISION_PROV = os.path.join(DATA_DIR, 'dft-road-casualty-statistics-collision-provisional-2025.csv')
CASUALTY_PROV  = os.path.join(DATA_DIR, 'dft-road-casualty-statistics-casualty-provisional-2025.csv')
VEHICLE_PROV   = os.path.join(DATA_DIR, 'dft-road-casualty-statistics-vehicle-provisional-2025.csv')
LA_LOOKUP      = os.path.join(DATA_DIR, 'local-authority-ons-district-names.csv')
POP_FILE       = os.path.join(DATA_DIR, 'ons_la_population_2024.csv')
POP_SKIPROWS   = 7
POP_CODE_COL   = 'mnemonic'
POP_VAL_COL    = '2024'

DTYPE = {'collision_index': str, 'collision_ref_no': str}

def savefig(name):
    """Save current figure to outputs/ folder."""
    path = os.path.join(OUTPUT_DIR, f'{name}.png')
    plt.savefig(path, dpi=120, bbox_inches='tight')
    print(f'Saved: {path}')

print('DATA_DIR   :', DATA_DIR,    '| exists:', os.path.exists(DATA_DIR))
print('CLEANED_DIR:', CLEANED_DIR, '| exists:', os.path.exists(CLEANED_DIR))
print('OUTPUT_DIR :', OUTPUT_DIR,  '| exists:', os.path.exists(OUTPUT_DIR))
print('PyArrow available:', HAS_ARROW)


# ── 1. Data Load and Preparation ──────────────────────────────────────────────

def load_filtered(filepath, year_start, year_end, dtype=None):
    """Load CSV filtered to year window. Uses PyArrow if available, chunked fallback otherwise."""
    if HAS_ARROW:
        table = pa_csv.read_csv(filepath)
        mask  = (pc.greater_equal(table['collision_year'], year_start) &
                 pc.less_equal(table['collision_year'], year_end))
        df = table.filter(mask).to_pandas()
        df.columns = df.columns.str.lower()
        return df
    else:
        chunks = []
        for chunk in pd.read_csv(filepath, dtype=dtype, low_memory=False, chunksize=100_000):
            chunk.columns = chunk.columns.str.lower()
            chunk = chunk[chunk['collision_year'].between(year_start, year_end)]
            chunks.append(chunk)
        return pd.concat(chunks, ignore_index=True)

col_hist = load_filtered(COLLISION_HIST, YEAR_START, YEAR_END, DTYPE)
cas_hist = load_filtered(CASUALTY_HIST,  YEAR_START, YEAR_END, DTYPE)
veh_hist = load_filtered(VEHICLE_HIST,   YEAR_START, YEAR_END, DTYPE)

col_prov = pd.read_csv(COLLISION_PROV, dtype=DTYPE, low_memory=False)
cas_prov = pd.read_csv(CASUALTY_PROV,  dtype=DTYPE, low_memory=False)
veh_prov = pd.read_csv(VEHICLE_PROV,   dtype=DTYPE, low_memory=False)
for df in [col_prov, cas_prov, veh_prov]:
    df.columns = df.columns.str.lower()
    df['provisional'] = True

col_hist['provisional'] = False
cas_hist['provisional'] = False
veh_hist['provisional'] = False

col = pd.concat([col_hist, col_prov], ignore_index=True)
cas = pd.concat([cas_hist, cas_prov], ignore_index=True)
veh = pd.concat([veh_hist, veh_prov], ignore_index=True)

YEARS = sorted(col['collision_year'].dropna().unique().astype(int).tolist())
print(f'col: {len(col):,}  cas: {len(cas):,}  veh: {len(veh):,}')
print(f'Years: {YEARS}')

# Replace -1 with NaN
CODED_COL = ['collision_severity','enhanced_severity_collision','speed_limit','road_type',
             'junction_detail','junction_control','light_conditions','urban_or_rural_area',
             'police_force','collision_injury_based','day_of_week','weather_conditions',
             'road_surface_conditions']
CODED_CAS = ['casualty_severity','enhanced_casualty_severity','casualty_type','casualty_class',
             'age_of_casualty','age_band_of_casualty','sex_of_casualty','casualty_imd_decile',
             'casualty_injury_based']
CODED_VEH = ['vehicle_type','sex_of_driver','age_of_driver','age_band_of_driver',
             'journey_purpose_of_driver','driver_imd_decile','propulsion_code']

for fields, df in [(CODED_COL, col), (CODED_CAS, cas), (CODED_VEH, veh)]:
    for f in fields:
        if f in df.columns:
            df[f] = pd.to_numeric(df[f], errors='coerce')
            df.loc[df[f] < 0, f] = np.nan

col['date']  = pd.to_datetime(col['date'], dayfirst=True, errors='coerce')
col['month'] = col['date'].dt.month
col['ksi']   = col['collision_severity'].isin([1, 2])
col['fatal'] = col['collision_severity'] == 1

# Label maps
ROAD_USER_MAP = {
    0:  'Pedestrian',
    1:  'Cyclist',
    2:  'Motorcyclist',        # <=50cc
    3:  'Motorcyclist',        # <=125cc
    4:  'Motorcyclist',        # 125-500cc
    5:  'Motorcyclist',        # >500cc
    8:  'Car occupant',        # taxi/PHV
    9:  'Car occupant',
    10: 'Car occupant',        # minibus
    11: 'Bus/coach occupant',
    16: 'Other',               # horse rider
    17: 'Other',               # agricultural
    18: 'Other',               # tram
    19: 'Van/goods occupant',  # <=3.5t
    20: 'Van/goods occupant',  # 3.5-7.5t
    21: 'Van/goods occupant',  # >=7.5t
    22: 'Motorcyclist',        # electric motorcycle
    23: 'Motorcyclist',        # unknown cc
    90: 'Other',
    97: 'Motorcyclist',
    98: 'Van/goods occupant',  # unknown weight
    99: 'Other',
}
ROAD_USER_ORDER = ['Pedestrian','Cyclist','Motorcyclist','Car occupant',
                   'Bus/coach occupant','Van/goods occupant','Other']

URBAN_MAP    = {1:'Urban', 2:'Rural', 3:'Unallocated'}
ROAD_MAP     = {1:'Roundabout', 2:'One way street', 3:'Dual carriageway',
                6:'Single carriageway', 7:'Slip road', 9:'Unknown', 12:'One way/Slip road'}
JUNCTION_MAP = {0:'Not at junction', 13:'T or Y junction', 16:'Crossroads',
                17:'More than 4 arms', 18:'Private drive or entrance',
                19:'Other junction', 99:'Unknown'}
DAY_MAP      = {1:'Sun', 2:'Mon', 3:'Tue', 4:'Wed', 5:'Thu', 6:'Fri', 7:'Sat'}
FORCE_MAP    = {1:'Metropolitan Police',3:'Cumbria',4:'Lancashire',5:'Merseyside',
                6:'Greater Manchester',7:'Cheshire',10:'Humberside',11:'West Yorkshire',
                12:'South Yorkshire',13:'West Midlands',14:'East Midlands',16:'Staffordshire',
                17:'West Mercia',20:'Thames Valley',21:'Hampshire',22:'South East',
                23:'Essex',24:'Norfolk',25:'Suffolk',26:'Bedfordshire',27:'Hertfordshire',
                30:'Cambridgeshire',31:'Northamptonshire',32:'Leicestershire',33:'Warwickshire',
                34:'Avon and Somerset',35:'Gloucestershire',36:'Wiltshire',37:'Dorset',
                38:'Devon and Cornwall',39:'Sussex',40:'Kent',41:'Surrey',42:'Northumbria',
                43:'Durham',44:'North Yorkshire',45:'Dyfed-Powys',46:'Gwent',47:'South Wales',
                48:'North Wales',50:'Lincolnshire',52:'Cleveland',53:'City of London',
                54:'Nottinghamshire',55:'Derbyshire',99:'Unknown/Other'}

cas['road_user']       = cas['casualty_type'].map(ROAD_USER_MAP).fillna('Other')
col['ur_label']        = col['urban_or_rural_area'].map(URBAN_MAP)
col['road_type_label'] = col['road_type'].map(ROAD_MAP)
col['junction_label']  = col['junction_detail'].map(JUNCTION_MAP)
col['day_label']       = col['day_of_week'].map(DAY_MAP)
col['force_name']      = col['police_force'].map(FORCE_MAP).fillna('Unknown')

la_lookup = pd.read_csv(LA_LOOKUP)[['code','la_name']].rename(
    columns={'code':'local_authority_ons_district'})
col = col.merge(la_lookup, on='local_authority_ons_district', how='left')

cas_col = cas.merge(col, on=['collision_index','collision_year'], how='left')
full    = cas_col.merge(veh, on=['collision_index','collision_year','vehicle_reference'], how='left')
assert len(full) == len(cas), f'Merge row count mismatch: {len(full)} vs {len(cas)}'

print('Cleaning, label maps, and merge complete.')
print(f'col: {col.shape}  cas: {cas.shape}  veh: {veh.shape}  full: {full.shape}')


# ── 2. Exploratory Analysis and Descriptive Checks ────────────────────────────
# IBRS adoption is the key data quality issue for KSI trend interpretation.
# Raw KSI is used as the primary outcome; adjusted series shown for context only.

NON_NEGATIVE_CHECK = CODED_COL + CODED_CAS + CODED_VEH

def missingness(df, fields):
    rows = []
    for f in fields:
        if f not in df.columns:
            rows.append({'Field': f, 'Missing (n)': '-', 'Missing (%)': '-'})
            continue
        null_mask = df[f].isna()
        if f in NON_NEGATIVE_CHECK:
            null_mask = null_mask | (pd.to_numeric(df[f], errors='coerce') < 0)
        n   = null_mask.sum()
        pct = n / len(df) * 100
        flag = '  *** HIGH ***' if pct > 5 else ''
        rows.append({'Field': f, 'Missing (n)': f'{n:,}', 'Missing (%)': f'{pct:.1f}%{flag}'})
    return pd.DataFrame(rows)

key_col = ['collision_severity','enhanced_severity_collision','date','latitude','longitude',
           'speed_limit','road_type','junction_detail','light_conditions','urban_or_rural_area',
           'collision_injury_based','collision_adjusted_severity_serious']
print('=== Collision table missingness ===')
print(missingness(col, key_col).to_string(index=False))

ibrs = (col.groupby('collision_year')['collision_injury_based']
        .apply(lambda x: (x==1).sum()/len(x)*100).reset_index())
ibrs.columns = ['year','ibrs_pct']
ibrs['ibrs_pct'] = ibrs['ibrs_pct'].round(1)

fig, ax = plt.subplots(figsize=(10,4))
colors = ['#e74c3c' if y in COVID_YEARS else '#27ae60' for y in ibrs['year']]
bars = ax.bar(ibrs['year'], ibrs['ibrs_pct'], color=colors, width=0.6, edgecolor='white')
ax.axhline(90, color='#2c3e50', linestyle='--', linewidth=1, label='90% threshold')
for bar, val in zip(bars, ibrs['ibrs_pct']):
    ax.text(bar.get_x()+bar.get_width()/2, val+1, f'{val:.0f}%', ha='center', fontsize=8)
ax.set_ylim(0, 108)
ax.set_xlabel('Year'); ax.set_ylabel('% IBRS-recorded')
ax.set_title('Fig 1: IBRS adoption by year', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show(); savefig('fig_01_ibrs_adoption')
print(ibrs.to_string(index=False))

sev = col.groupby(['collision_year','collision_severity']).size().unstack(fill_value=0)
sev = sev.rename(columns={1:'Fatal',2:'Serious',3:'Slight'})
sev['KSI']   = sev['Fatal'] + sev['Serious']
sev['Total'] = sev[['Fatal','Serious','Slight']].sum(axis=1)
sev['Serious/Slight'] = (sev['Serious']/sev['Slight']).round(3)
sev['KSI%'] = (sev['KSI']/sev['Total']*100).round(1)
print('Severity breakdown by year:')
print(sev[['Fatal','Serious','Slight','KSI','Serious/Slight','KSI%']].to_string())
sev[['Fatal','Serious','Slight','KSI','Serious/Slight','KSI%']].to_csv(
    os.path.join(OUTPUT_DIR,'tab_severity_by_year.csv'))
print('Note: rising Serious/Slight ratio reflects IBRS adoption, not genuine change.')

force_yr = col.groupby(['collision_year','police_force']).size().unstack(fill_value=0)
forces_per_year = (force_yr > 0).sum(axis=1)
print('Forces reporting per year:')
print(forces_per_year.to_string())

valid = col['latitude'].notna() & col['longitude'].notna() & (col['latitude'] != 0)
in_gb = valid & col['latitude'].between(49.5,61.0) & col['longitude'].between(-8.0,2.0)
print(f'\nValid coordinates : {valid.sum():,}  ({valid.sum()/len(col)*100:.1f}%)')
print(f'Within GB bounds  : {in_gb.sum():,}  ({in_gb.sum()/len(col)*100:.1f}%)')


# ── 3. KSI Trends ─────────────────────────────────────────────────────────────

# Seasonality check — validate provisional year annualisation
prov_year      = max(y for y in YEARS if y > YEAR_END)
base_years_ann = [y for y in range(YEAR_END - 4, YEAR_END + 1) if y not in COVID_YEARS]
recent_all     = base_years_ann + [prov_year]

monthly = (col[col['collision_year'].isin(recent_all) & col['ksi']]
           .groupby(['collision_year','month']).size().reset_index(name='ksi_count'))
share = []
for yr in base_years_ann:
    d  = monthly[monthly['collision_year']==yr]
    if len(d) == 0: continue
    h1 = d[d['month']<=6]['ksi_count'].sum()
    share.append({'year':yr,'h1_pct':h1/d['ksi_count'].sum()*100})
share_df   = pd.DataFrame(share)
ann_factor = 100 / share_df['h1_pct'].mean()
ksi_h1     = monthly[monthly['collision_year']==prov_year]['ksi_count'].sum()
ksi_ann    = ksi_h1 * ann_factor

fig, ax = plt.subplots(figsize=(12,5))
palette_ann = plt.cm.get_cmap('tab10', len(recent_all))
for i, yr in enumerate(recent_all):
    d = monthly[monthly['collision_year']==yr]
    ax.plot(d['month'], d['ksi_count'], marker='o', label=str(yr),
            color=palette_ann(i),
            linestyle='--' if yr==prov_year else '-',
            linewidth=2.5 if yr >= YEAR_END - 2 else 1.5, markersize=5)
ax.axvline(6.5, color='grey', linestyle=':', linewidth=1.5, label=f'{prov_year} data ends')
ax.set_xticks(range(1,13))
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_title(f'Fig 2: Monthly KSI seasonality ({min(recent_all)}–{prov_year})', fontweight='bold')
ax.set_ylabel('KSI collisions')
ax.legend(title='Year', fontsize=8)
plt.tight_layout(); plt.show(); savefig('fig_02_seasonality')

print(f'Average H1 share (excl. COVID): {share_df["h1_pct"].mean():.1f}%')
print(f'{prov_year} Jan–Jun KSI              : {ksi_h1:,}')
print(f'{prov_year} annualised estimate      : {ksi_ann:,.0f}  (provisional — treat with caution)')

annual_nat = (col[col['collision_year']<=YEAR_END]
              .groupby('collision_year')
              .agg(collisions=('collision_index','nunique'),
                   ksi=('ksi','sum'), fatals=('fatal','sum'))
              .reset_index())

fig, axes = plt.subplots(2,1,figsize=(14,10))
ax = axes[0]
ax.fill_between(annual_nat['collision_year'], annual_nat['ksi'], alpha=0.10, color='#e67e22')
ax.plot(annual_nat['collision_year'], annual_nat['ksi'],    color='#e67e22', lw=2, marker='o', label='KSI (raw)')
ax.plot(annual_nat['collision_year'], annual_nat['fatals'], color='#c0392b', lw=2, marker='s', label='Fatal')
ax.scatter([prov_year],[ksi_ann], color='#e67e22', marker='D', s=70, zorder=5,
           label=f'{prov_year} annualised ({ksi_ann:,.0f}) — provisional')
ax.axvspan(min(COVID_YEARS)-0.5, max(COVID_YEARS)+0.5, alpha=0.08, color='grey', label='COVID period')
ax.axvline(2016, color='#8e44ad', lw=1.5, linestyle='--', alpha=0.7, label='IBRS rollout begins')
ax.set_title(f'KSI and fatal collisions, {YEAR_START}–{prov_year}', fontweight='bold')
ax.set_ylabel('Collisions')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.legend(fontsize=8)

ax2 = axes[1]
by_sev = (col[col['collision_year'].between(YEAR_START,YEAR_END)]
          .groupby(['collision_year','collision_severity']).size().unstack(fill_value=0)
          .rename(columns={1:'Fatal',2:'Serious',3:'Slight'}))
bottom = np.zeros(len(by_sev))
for sev, color in [('Slight','#3498db'),('Serious','#e67e22'),('Fatal','#c0392b')]:
    if sev in by_sev.columns:
        ax2.bar(by_sev.index, by_sev[sev], bottom=bottom, color=color,
                label=sev, edgecolor='white', width=0.7)
        bottom += by_sev[sev].values
ax2.axvspan(min(COVID_YEARS)-0.5, max(COVID_YEARS)+0.5, alpha=0.08, color='grey')
ax2.axvline(2016, color='#8e44ad', lw=1.5, linestyle='--', alpha=0.7)
ax2.set_title('Collision severity breakdown by year', fontweight='bold')
ax2.set_ylabel('Collisions')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax2.legend()
plt.suptitle(f'Fig 3: KSI trend {YEAR_START}–{YEAR_END}', fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show(); savefig('fig_03_ksi_trend')

# Raw vs IBRS-adjusted KSI — context only; adjusted series is partial (IBRS records only)
ibrs_start = 2016
ibrs_w = col[(col['collision_year']>=ibrs_start) & (col['collision_year']<=YEAR_END) &
             (~col['provisional'])].copy()
adj = ibrs_w.groupby('collision_year').agg(
    raw_fatal   = ('fatal','sum'),
    raw_serious = ('collision_severity', lambda x: (x==2).sum()),
    adj_serious = ('collision_adjusted_severity_serious','sum'),
    ibrs_pct    = ('collision_injury_based', lambda x: (x==1).sum()/len(x)*100)
).reset_index()
adj['raw_ksi'] = adj['raw_fatal'] + adj['raw_serious']
adj['adj_ksi'] = adj['raw_fatal'] + adj['adj_serious']
adj['ibrs_pct'] = adj['ibrs_pct'].round(1)

fig, axes = plt.subplots(1,2,figsize=(14,5))
ax = axes[0]
ax.plot(adj['collision_year'], adj['raw_ksi'], marker='o', color='#e67e22', lw=2, label='Raw KSI')
ax.plot(adj['collision_year'], adj['adj_ksi'], marker='s', color='#2ecc71', lw=2,
        linestyle='--', label='Adjusted KSI (partial — IBRS records only)')
ax.fill_between(adj['collision_year'], adj['raw_ksi'], adj['adj_ksi'], alpha=0.1, color='grey')
for yr in COVID_YEARS:
    ax.axvspan(yr-0.4,yr+0.4, alpha=0.1, color='grey')
ax.set_title(f'Raw vs adjusted KSI ({ibrs_start}–{YEAR_END})', fontweight='bold')
ax.set_ylabel('Collisions')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.legend()

ax2 = axes[1]
gap = adj['adj_ksi'] - adj['raw_ksi']
ax2.bar(adj['collision_year'], gap,
        color=['#c0392b' if d>0 else '#2ecc71' for d in gap], width=0.6, edgecolor='white')
ax2.axhline(0, color='black', lw=0.8)
ax2.set_title('Adjusted minus raw KSI per year', fontweight='bold')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
plt.suptitle('Fig 4: Effect of severity adjustment on KSI series\n'
             'Note: adjusted series is partial — only IBRS-recorded collisions contribute',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show(); savefig('fig_04_raw_vs_adjusted')

adj[['collision_year','raw_serious','adj_serious','raw_ksi','adj_ksi','ibrs_pct']].to_csv(
    os.path.join(OUTPUT_DIR,'tab_raw_vs_adjusted_ksi.csv'), index=False)
print(adj[['collision_year','raw_ksi','adj_ksi','ibrs_pct']].to_string(index=False))


# ── 4. Road User Severity Profile ─────────────────────────────────────────────

prov_map = col.drop_duplicates('collision_index').set_index('collision_index')['provisional']
cas_w = cas.copy()
cas_w['provisional'] = cas_w['collision_index'].map(prov_map).fillna(False)
window = cas_w[(cas_w['collision_year']>=ibrs_start) & (cas_w['collision_year']<=YEAR_END) &
               (~cas_w['provisional'])].copy()

ru = window.groupby('road_user')['casualty_severity'].value_counts().unstack(fill_value=0)
ru = ru.rename(columns={1:'Fatal',2:'Serious',3:'Slight'})
for c in ['Fatal','Serious','Slight']:
    if c not in ru.columns: ru[c] = 0
ru['Total'] = ru['Fatal']+ru['Serious']+ru['Slight']
ru['KSI']   = ru['Fatal']+ru['Serious']
ru['KSI%']  = (ru['KSI']/ru['Total']*100).round(1)
ru['Fatal%'] = (ru['Fatal']/ru['Total']*100).round(2)
ru = ru.sort_values('Total', ascending=False)

fig, axes = plt.subplots(1,2,figsize=(14,6))
ax = axes[0]
top = ru[ru['Total']>1000].copy()
bottom = np.zeros(len(top))
for sev, color in [('Slight','#3498db'),('Serious','#e67e22'),('Fatal','#c0392b')]:
    ax.bar(range(len(top)), top[sev], bottom=bottom, color=color, label=sev, edgecolor='white')
    bottom += top[sev].values
ax.set_xticks(range(len(top)))
ax.set_xticklabels(top.index, rotation=35, ha='right', fontsize=9)
ax.set_title('Total casualties by road user type', fontweight='bold')
ax.set_ylabel('Casualties')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.legend()

ax2 = axes[1]
ksi_plot = ru[ru['Total']>1000].sort_values('KSI%', ascending=True)
bar_colors = ['#c0392b' if k>30 else '#e67e22' if k>20 else '#3498db' for k in ksi_plot['KSI%']]
ax2.barh(ksi_plot.index, ksi_plot['KSI%'], color=bar_colors, edgecolor='white', height=0.6)
ax2.axvline(20, color='grey', linestyle='--', lw=1, label='20% reference')
ax2.set_title('KSI rate by road user type (%)', fontweight='bold')
ax2.set_xlabel('KSI %')
ax2.legend(fontsize=8)
plt.suptitle(f'Fig 5: Casualty profile by road user type ({ibrs_start}–{YEAR_END})',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show(); savefig('fig_05_road_user_severity')

ru[['Fatal','Serious','Slight','Total','KSI','KSI%','Fatal%']].to_csv(
    os.path.join(OUTPUT_DIR,'tab_road_user_severity.csv'))
print(ru[['Fatal','Serious','Slight','Total','KSI','KSI%','Fatal%']].to_string())


# ── 5. Geographic Hotspot Analysis ────────────────────────────────────────────
# Two hotspot definitions: raw KSI count and KSI rate per collision.
# Recent window for operational charts; full window for rank stability check.

recent_start = min(RECENT_YEARS)
hot = col[(col['collision_year']>=recent_start) & (col['collision_year']<=YEAR_END)].copy()

la = hot.groupby('local_authority_ons_district').agg(
    collisions=('collision_index','nunique'), ksi=('ksi','sum'), fatals=('fatal','sum')
).reset_index()
la['ksi_pct']   = (la['ksi']/la['collisions']*100).round(1)
la['fatal_pct'] = (la['fatals']/la['collisions']*100).round(2)
la = la.merge(la_lookup, on='local_authority_ons_district', how='left')
la['la_name'] = la['la_name'].fillna(la['local_authority_ons_district'])
la = la.sort_values('ksi', ascending=False)

fig, axes = plt.subplots(1,2,figsize=(14,8))
top20 = la.head(20).sort_values('ksi')
axes[0].barh(range(len(top20)), top20['ksi'], color='#3498db', edgecolor='white')
axes[0].set_yticks(range(len(top20)))
axes[0].set_yticklabels(top20['la_name'], fontsize=8)
axes[0].set_title(f'Top 20 LAs — KSI count ({recent_start}–{YEAR_END})', fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

la_rate = (la[la['collisions']>200]
           .sort_values('ksi_pct', ascending=False)
           .head(20).sort_values('ksi_pct'))
axes[1].barh(range(len(la_rate)), la_rate['ksi_pct'], color='#c0392b', edgecolor='white')
axes[1].set_yticks(range(len(la_rate)))
axes[1].set_yticklabels(la_rate['la_name'], fontsize=8)
axes[1].set_title(f'Top 20 LAs — KSI rate % ({recent_start}–{YEAR_END})', fontweight='bold')
axes[1].set_xlabel('KSI %')
plt.suptitle(f'Fig 6: KSI by local authority ({recent_start}–{YEAR_END})',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show(); savefig('fig_06_la_hotspots')
la.to_csv(os.path.join(OUTPUT_DIR,'tab_la_hotspots.csv'), index=False)

try:
    import geopandas as gpd
    la_geo = gpd.read_file(CHOROPLETH_URL)
    la_geo = la_geo.merge(la, left_on='LAD24CD', right_on='local_authority_ons_district', how='left')
    fig, axes = plt.subplots(1,2,figsize=(16,12))
    for ax, col_name, title, cmap in [
        (axes[0],'ksi',f'KSI count ({recent_start}–{YEAR_END})','YlOrRd'),
        (axes[1],'ksi_pct',f'KSI rate % ({recent_start}–{YEAR_END})','OrRd'),
    ]:
        la_geo.plot(column=col_name, ax=ax, cmap=cmap, legend=True,
                    missing_kwds={'color':'#eeeeee'},
                    legend_kwds={'shrink':0.6,'label':title})
        ax.set_title(title, fontweight='bold', fontsize=12)
        ax.set_axis_off()
    plt.suptitle(f'Fig 6b: KSI hotspot choropleth ({recent_start}–{YEAR_END})',
                 fontweight='bold', fontsize=13)
    plt.tight_layout(); plt.show(); savefig('fig_06b_choropleth')
except ImportError:
    print('geopandas not available — choropleth skipped.')

# Full-window rank comparison: count vs severity rate
full_window = full[
    full['collision_year'].between(YEAR_START, YEAR_END) & (~full['provisional'])
].copy()
col_window = full_window.drop_duplicates(subset=['collision_index','collision_year'])

la_full = col_window.groupby('la_name').agg(
    total_collisions = ('collision_index','nunique'),
    ksi_collisions   = ('ksi','sum'),
    fatals           = ('fatal','sum')
).reset_index()
ksi_cas_count = (full_window[full_window['ksi']]
                 .groupby('la_name').size().reset_index(name='ksi_casualties_total'))
la_full = la_full.merge(ksi_cas_count, on='la_name', how='left')
la_full['severity_rate']      = la_full['ksi_collisions'] / la_full['total_collisions'] * 100
la_full['rank_raw_burden']    = la_full['ksi_casualties_total'].rank(ascending=False, method='first').astype(int)
la_full['rank_severity_rate'] = la_full['severity_rate'].rank(ascending=False, method='first').astype(int)

la_full_f = la_full[la_full['total_collisions'] >= MIN_COLLISIONS].copy()
la_full_f['rank_shift'] = abs(la_full_f['rank_raw_burden'] - la_full_f['rank_severity_rate'])
la_full_f['flagged']    = la_full_f['rank_shift'] >= 5

top_n_count = set(la_full_f.nsmallest(HOTSPOT_TOP_N,'rank_raw_burden')['la_name'])
top_n_rate  = set(la_full_f.nsmallest(HOTSPOT_TOP_N,'rank_severity_rate')['la_name'])
overlap     = top_n_count & top_n_rate
count_only  = top_n_count - top_n_rate
rate_only   = top_n_rate  - top_n_count

print(f'=== Robustness check: hotspot definition sensitivity ({YEAR_START}–{YEAR_END}) ===')
print(f'In both lists ({len(overlap)}/{HOTSPOT_TOP_N}) : {sorted(overlap)}')
print(f'Count top {HOTSPOT_TOP_N} only ({len(count_only)}) : {sorted(count_only)}')
print(f'Rate top {HOTSPOT_TOP_N} only  ({len(rate_only)})  : {sorted(rate_only)}')

fig, ax = plt.subplots(figsize=(9,7))
plot_data = la_full_f[
    (la_full_f['rank_raw_burden']<=30) | (la_full_f['rank_severity_rate']<=30)
].copy()
colors_scatter = ['#e74c3c' if f else '#3498db' for f in plot_data['flagged']]
ax.scatter(plot_data['rank_raw_burden'], plot_data['rank_severity_rate'],
           c=colors_scatter, s=50, zorder=3)
ax.plot([0,50],[0,50], linestyle='--', color='grey', linewidth=1)
for _, row in plot_data[plot_data['flagged']].iterrows():
    ax.annotate(row['la_name'], (row['rank_raw_burden'], row['rank_severity_rate']),
                fontsize=7, ha='left', va='bottom')
ax.set_xlabel(f'Rank by raw KSI count ({YEAR_START}–{YEAR_END})')
ax.set_ylabel(f'Rank by collision-adjusted severity rate ({YEAR_START}–{YEAR_END})')
ax.set_title('Fig 6c: Burden rank vs severity rate rank\nRed = rank shifts 5+ places',
             fontweight='bold')
plt.tight_layout(); plt.show(); savefig('fig_06c_count_vs_rate_rank')
la_full_f.sort_values('rank_raw_burden').to_csv(
    os.path.join(OUTPUT_DIR,'tab_la_full_window_hotspots.csv'), index=False)

# Three-way exposure robustness check
if os.path.exists(POP_FILE):
    ons_pop = pd.read_csv(POP_FILE, skiprows=POP_SKIPROWS)[[POP_CODE_COL, POP_VAL_COL]].rename(
        columns={POP_CODE_COL:'local_authority_ons_district', POP_VAL_COL:'population'})
    ons_pop['population'] = pd.to_numeric(ons_pop['population'], errors='coerce')
    ons_pop = ons_pop[ons_pop['local_authority_ons_district'].notna()]
    la_code = col_window[['la_name','local_authority_ons_district']].drop_duplicates().dropna()
    la_full_pop = (la_full_f
                   .merge(la_code, on='la_name', how='left')
                   .merge(ons_pop, on='local_authority_ons_district', how='left'))
    la_full_pop = la_full_pop[la_full_pop['population'].notna()].copy()
    la_full_pop['ksi_per_100k']    = la_full_pop['ksi_casualties_total'] / la_full_pop['population'] * 100000
    la_full_pop['rank_per_capita'] = la_full_pop['ksi_per_100k'].rank(ascending=False, method='first').astype(int)
    three_way = la_full_pop[
        (la_full_pop['rank_raw_burden']<=HOTSPOT_TOP_N) |
        (la_full_pop['rank_severity_rate']<=HOTSPOT_TOP_N) |
        (la_full_pop['rank_per_capita']<=HOTSPOT_TOP_N)
    ][['la_name','ksi_casualties_total','severity_rate','ksi_per_100k',
       'rank_raw_burden','rank_severity_rate','rank_per_capita']].copy()
    three_way['max_rank_shift'] = (
        three_way[['rank_raw_burden','rank_severity_rate','rank_per_capita']]
        .apply(lambda r: r.max()-r.min(), axis=1))
    three_way['flagged'] = three_way['max_rank_shift'] >= 5
    print('=== Three-way exposure robustness check ===')
    print(three_way.sort_values('rank_raw_burden')[
        ['la_name','rank_raw_burden','rank_severity_rate','rank_per_capita',
         'max_rank_shift','flagged']].to_string(index=False))
    three_way.to_csv(os.path.join(OUTPUT_DIR,'tab_three_way_exposure_robustness.csv'), index=False)
else:
    print(f'Population file not found at {POP_FILE} — three-way check skipped.')
    print('Download from Nomis (LA district/unitary as of April 2023) and save to Data/.')


# ── 6. Contributory Factors ───────────────────────────────────────────────────

ur = (hot.groupby('ur_label')['collision_severity']
      .value_counts().unstack(fill_value=0)
      .rename(columns={1:'Fatal',2:'Serious',3:'Slight'}))
for c in ['Fatal','Serious','Slight']:
    if c not in ur.columns: ur[c] = 0
ur['Total']  = ur['Fatal']+ur['Serious']+ur['Slight']
ur['KSI']    = ur['Fatal']+ur['Serious']
ur['KSI%']   = (ur['KSI']/ur['Total']*100).round(1)
ur['Fatal%'] = (ur['Fatal']/ur['Total']*100).round(2)
print(f'Urban vs Rural KSI breakdown ({recent_start}–{YEAR_END}):')
print(ur[['Fatal','Serious','Slight','Total','KSI','KSI%','Fatal%']].to_string())
ur[['Fatal','Serious','Slight','Total','KSI','KSI%','Fatal%']].to_csv(
    os.path.join(OUTPUT_DIR,'tab_urban_rural.csv'))

fig, axes = plt.subplots(2,2,figsize=(14,10))
speed = (hot[hot['speed_limit'].notna()]
         .groupby('speed_limit')['collision_severity']
         .value_counts().unstack(fill_value=0)
         .rename(columns={1:'Fatal',2:'Serious',3:'Slight'}))
for c in ['Fatal','Serious','Slight']:
    if c not in speed.columns: speed[c] = 0
speed['KSI']   = speed['Fatal']+speed['Serious']
speed['Total'] = speed['Fatal']+speed['Serious']+speed['Slight']
speed['KSI%']  = (speed['KSI']/speed['Total']*100).round(1)

s = speed.sort_values('KSI', ascending=True)
axes[0,0].barh([f'{int(x)} mph' for x in s.index], s['KSI'],
               color='#e67e22', edgecolor='white', height=0.6)
axes[0,0].set_title('KSI by speed limit — count', fontweight='bold')
axes[0,0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

sr = speed.sort_values('KSI%', ascending=True)
axes[0,1].barh([f'{int(x)} mph' for x in sr.index], sr['KSI%'],
               color='#c0392b', edgecolor='white', height=0.6)
axes[0,1].set_title('KSI rate by speed limit — %', fontweight='bold')

rc = (hot[hot['road_type_label'].notna()]
      .groupby('road_type_label')['collision_severity']
      .value_counts().unstack(fill_value=0)
      .rename(columns={1:'Fatal',2:'Serious',3:'Slight'}))
for c in ['Fatal','Serious','Slight']:
    if c not in rc.columns: rc[c] = 0
rc['KSI']   = rc['Fatal']+rc['Serious']
rc['Total'] = rc['Fatal']+rc['Serious']+rc['Slight']
rc['KSI%']  = (rc['KSI']/rc['Total']*100).round(1)

r = rc.sort_values('KSI', ascending=True)
axes[1,0].barh(r.index, r['KSI'], color='#3498db', edgecolor='white', height=0.6)
axes[1,0].set_title('KSI by road type — count', fontweight='bold')
axes[1,0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

rr = rc.sort_values('KSI%', ascending=True)
axes[1,1].barh(rr.index, rr['KSI%'], color='#9b59b6', edgecolor='white', height=0.6)
axes[1,1].set_title('KSI rate by road type — %', fontweight='bold')
plt.suptitle(f'Fig 7: KSI by speed limit and road type ({recent_start}–{YEAR_END})',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show(); savefig('fig_07_speed_road_type')

fig, axes = plt.subplots(1,2,figsize=(13,4.5))
day_order = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
day = (hot[hot['day_label'].notna() & hot['ksi']].groupby('day_label').size().reindex(day_order))
axes[0].bar(day.index, day.values, color='#9b59b6', edgecolor='white', width=0.6)
axes[0].set_title('KSI by day of week', fontweight='bold')
axes[0].set_ylabel('KSI collisions')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

hot_h = hot[hot['ksi']].copy()
hot_h['hour'] = pd.to_datetime(hot_h['time'], format='%H:%M', errors='coerce').dt.hour
hour = hot_h.groupby('hour').size()
axes[1].plot(hour.index, hour.values, color='#e74c3c', lw=2, marker='o', markersize=4)
axes[1].set_title('KSI by hour of day', fontweight='bold')
axes[1].set_xlabel('Hour'); axes[1].set_ylabel('KSI collisions')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
plt.suptitle(f'Fig 8: KSI time patterns ({recent_start}–{YEAR_END})',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show(); savefig('fig_08_time_patterns')

jn = hot[hot['junction_detail'].notna()].copy()
jn['junction_label'] = jn['junction_detail'].astype(int).map(JUNCTION_MAP)
jn = jn[jn['junction_label'].notna()]
jn_grp = (jn.groupby('junction_label')['collision_severity']
          .value_counts().unstack(fill_value=0)
          .rename(columns={1:'Fatal',2:'Serious',3:'Slight'}))
for c in ['Fatal','Serious','Slight']:
    if c not in jn_grp.columns: jn_grp[c] = 0
jn_grp['KSI']   = jn_grp['Fatal']+jn_grp['Serious']
jn_grp['Total'] = jn_grp['Fatal']+jn_grp['Serious']+jn_grp['Slight']
jn_grp['KSI%']  = (jn_grp['KSI']/jn_grp['Total']*100).round(1)
jn_grp = jn_grp.sort_values('KSI', ascending=False)

fig, axes = plt.subplots(1,2,figsize=(14,5))
jn_count = jn_grp.sort_values('KSI', ascending=True)
axes[0].barh(jn_count.index, jn_count['KSI'], color='#3498db', edgecolor='white', height=0.6)
axes[0].set_title('KSI by junction type — count', fontweight='bold')
axes[0].set_xlabel('KSI collisions')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
jn_rate = jn_grp.sort_values('KSI%', ascending=True)
axes[1].barh(jn_rate.index, jn_rate['KSI%'], color='#9b59b6', edgecolor='white', height=0.6)
axes[1].set_title('KSI rate by junction type — %', fontweight='bold')
axes[1].set_xlabel('KSI %')
plt.suptitle(f'Fig 9: KSI by junction type ({recent_start}–{YEAR_END})',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show(); savefig('fig_09_junction_type')
jn_grp[['Fatal','Serious','Slight','Total','KSI','KSI%']].to_csv(
    os.path.join(OUTPUT_DIR,'tab_junction_type.csv'))
print(jn_grp[['Fatal','Serious','Slight','Total','KSI','KSI%']].to_string())


# ── 7. Local Authority Trend Analysis ─────────────────────────────────────────
# OLS per LA with 95% CIs. R² threshold distinguishes linear from non-linear trends.
# Three-year window comparison used as secondary cross-check.
# Built from full dataframe in memory — no external file dependency.

ksi = full[
    full['ksi'].eq(True) &
    full['provisional'].eq(False) &
    full['la_name'].notna()
].copy()
# Standardise road_user labels using ROAD_USER_MAP applied to casualty_type
ksi['road_user'] = ksi['casualty_type'].map(ROAD_USER_MAP).fillna('Other')

print(f'KSI casualties: {len(ksi):,}')
print(f'Years: {sorted(ksi["collision_year"].dropna().unique().astype(int).tolist())}')

def complete_las(df, years):
    return set(
        df[df['collision_year'].isin(years)]
        .groupby('la_name')['collision_year'].nunique()
        .pipe(lambda s: s[s==len(years)].index)
    )

complete     = complete_las(ksi, BASE_YEARS) & complete_las(ksi, RECENT_YEARS)
ksi_complete = ksi[ksi['la_name'].isin(complete)].copy()
print(f'LAs with complete data : {len(complete)}')
print(f'LAs excluded           : {ksi["la_name"].nunique() - len(complete)}')

annual = (ksi_complete.groupby(['la_name','collision_year'])
          .size().reset_index(name='KSI'))

ols_rows = []
for la, grp in annual.groupby('la_name'):
    grp = grp.dropna(subset=['KSI','collision_year'])
    if len(grp) < MIN_YEARS: continue
    x = grp['collision_year'].values
    y = grp['KSI'].values
    slope, intercept, r, p, se = stats.linregress(x, y)
    t_crit   = stats.t.ppf(0.975, df=len(x)-2)
    ci_lower = slope - t_crit * se
    ci_upper = slope + t_crit * se
    ols_rows.append({
        'la_name'  : la, 'slope': slope, 'ci_lower': ci_lower, 'ci_upper': ci_upper,
        'se': se, 'intercept': intercept, 'r_squared': r**2, 'p_value': p,
        'n_years': len(grp), 'mean_ksi': y.mean(),
    })

ols = pd.DataFrame(ols_rows)
ols = ols[ols['mean_ksi'] >= MIN_KSI_BASE].copy()
ols['trend_shape']      = ols['r_squared'].apply(lambda r: 'Linear' if r>=LOW_R2 else 'Non-linear')
ols['ci_includes_zero'] = (ols['ci_lower'] < 0) & (ols['ci_upper'] > 0)

def window_avg(df, years):
    return (df[df['collision_year'].isin(years)].groupby('la_name')
            .size().div(len(years)).reset_index(name='ksi_avg'))

base    = window_avg(ksi_complete, BASE_YEARS).rename(columns={'ksi_avg':'base_avg'})
recent  = window_avg(ksi_complete, RECENT_YEARS).rename(columns={'ksi_avg':'recent_avg'})
avg_chg = base.merge(recent, on='la_name')
avg_chg['avg_pct_change'] = (avg_chg['recent_avg']-avg_chg['base_avg'])/avg_chg['base_avg']*100
ols = ols.merge(avg_chg[['la_name','base_avg','recent_avg','avg_pct_change']], on='la_name', how='left')
ols['methods_agree'] = (ols['slope']>0) == (ols['avg_pct_change']>0)

# Flag authorities where OLS and 3-year comparison disagree on direction
direction_disagreements = ols[~ols['methods_agree']][['la_name','slope','avg_pct_change']].copy()
if len(direction_disagreements) > 0:
    print(f'\n=== Direction disagreements: OLS slope vs 3-year comparison ({len(direction_disagreements)} LAs) ===')
    print(direction_disagreements.round(2).to_string(index=False))
    print('These authorities have non-linear or spike patterns — treat trend conclusions with additional caution.')

print(f'\nLAs fitted           : {len(ols)}')
print(f'Linear (R² >= {LOW_R2}): {(ols["r_squared"]>=LOW_R2).sum()}')
print(f'CI includes zero     : {ols["ci_includes_zero"].sum()} LAs')
ols.to_csv(os.path.join(OUTPUT_DIR,'tab_la_ols_results.csv'), index=False)

worsening     = ols[ols['slope']>0].sort_values('slope', ascending=False).reset_index(drop=True)
worsening['rank'] = worsening.index + 1
top5_w        = worsening.head(TOP_N)
top5_w_names  = top5_w['la_name'].tolist()
top3_w_names  = top5_w_names[:TOP_N_LINES]

print(f'=== Top {TOP_N} worsening LAs ===')
print(top5_w[['rank','la_name','slope','ci_lower','ci_upper','r_squared','p_value',
              'trend_shape','avg_pct_change','ci_includes_zero']].round(3).to_string(index=False))

years_range = np.array(sorted(ksi_complete['collision_year'].unique()))
palette     = ['#e74c3c','#2980b9','#27ae60']
la_colors_w = dict(zip(top3_w_names, palette))

fig, ax = plt.subplots(figsize=(13,5))
ax.axvspan(min(COVID_YEARS)-0.5, max(COVID_YEARS)+0.5, alpha=0.10, color='grey')
ax.text(np.mean(COVID_YEARS), 1, 'Covid-19', ha='center', fontsize=8, color='grey',
        transform=ax.get_xaxis_transform())
for la in top3_w_names:
    sub = annual[annual['la_name']==la]
    row = ols[ols['la_name']==la].iloc[0]
    c   = la_colors_w[la]
    ci_str = f'CI [{row["ci_lower"]:.1f}, {row["ci_upper"]:.1f}]'
    ax.plot(sub['collision_year'], sub['KSI'], color=c, lw=2, marker='o', ms=5,
            label=f"{la} (slope={row['slope']:.1f}/yr, {ci_str}, R²={row['r_squared']:.2f})")
    ax.plot(years_range, row['intercept']+row['slope']*years_range,
            color=c, lw=1.2, linestyle='--', alpha=0.6)
ax.set_title(f'KSI trend — top {TOP_N_LINES} worsening LAs ({YEAR_START}–{YEAR_END})', fontsize=10)
ax.set_ylabel('KSI casualties')
ax.set_xticks(range(YEAR_START, YEAR_END+1)); ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.legend(fontsize=8)
plt.suptitle('Fig 10: Worsening LA trends with 95% CIs', fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show(); savefig('fig_10_worsening_las')

improving     = ols[ols['slope']<0].sort_values('slope').reset_index(drop=True)
improving['rank'] = improving.index + 1
top5_i        = improving.head(TOP_N)
top5_i_names  = top5_i['la_name'].tolist()
top3_i_names  = top5_i_names[:TOP_N_LINES]

print(f'=== Top {TOP_N} improving LAs ===')
print(top5_i[['rank','la_name','slope','ci_lower','ci_upper','r_squared','p_value',
              'trend_shape','avg_pct_change','ci_includes_zero']].round(3).to_string(index=False))

la_colors_i = dict(zip(top3_i_names, palette))
fig, ax = plt.subplots(figsize=(13,5))
ax.axvspan(min(COVID_YEARS)-0.5, max(COVID_YEARS)+0.5, alpha=0.10, color='grey')
ax.text(np.mean(COVID_YEARS), 1, 'Covid-19', ha='center', fontsize=8, color='grey',
        transform=ax.get_xaxis_transform())
for la in top3_i_names:
    sub = annual[annual['la_name']==la]
    row = ols[ols['la_name']==la].iloc[0]
    c   = la_colors_i[la]
    ci_str = f'CI [{row["ci_lower"]:.1f}, {row["ci_upper"]:.1f}]'
    ax.plot(sub['collision_year'], sub['KSI'], color=c, lw=2, marker='o', ms=5,
            label=f"{la} (slope={row['slope']:.1f}/yr, {ci_str}, R²={row['r_squared']:.2f})")
    ax.plot(years_range, row['intercept']+row['slope']*years_range,
            color=c, lw=1.2, linestyle='--', alpha=0.6)
ax.set_title(f'KSI trend — top {TOP_N_LINES} improving LAs ({YEAR_START}–{YEAR_END})', fontsize=10)
ax.set_ylabel('KSI casualties')
ax.set_xticks(range(YEAR_START, YEAR_END+1)); ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.legend(fontsize=8)
plt.suptitle('Fig 11: Improving LA trends with 95% CIs', fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show(); savefig('fig_11_improving_las')

# Factor breakdown — base vs recent, share AND absolute count
FACTORS = [
    ('road_user','Road user type'), ('ur_label','Urban / Rural'),
    ('road_type_label','Road type'), ('junction_label','Junction type'),
    ('speed_limit','Speed limit'),
]
base_label   = f'Base ({min(BASE_YEARS)}-{str(max(BASE_YEARS))[2:]})'
recent_label = f'Recent ({min(RECENT_YEARS)}-{str(max(RECENT_YEARS))[2:]})'

for grp_name, priority_names, title_suffix in [
    ('worsening', top5_w_names, 'worsening LAs'),
    ('improving', top5_i_names, 'improving LAs'),
]:
    deep = ksi_complete[
        ksi_complete['la_name'].isin(priority_names) &
        ksi_complete['collision_year'].isin(BASE_YEARS + RECENT_YEARS)
    ].copy()
    deep['window'] = deep['collision_year'].apply(
        lambda y: base_label if y in BASE_YEARS else recent_label)
    for col_name, label in FACTORS:
        if col_name not in deep.columns: continue
        grp = (deep[deep[col_name].notna()]
               .groupby(['la_name','window',col_name]).size().reset_index(name='KSI'))
        grp['pct'] = grp.groupby(['la_name','window'])['KSI'].transform(lambda x: x/x.sum()*100)
        cats = sorted(grp[col_name].dropna().unique().astype(str))
        cmap = plt.cm.get_cmap('Set2', len(cats))
        cat_colors = {c: cmap(i) for i, c in enumerate(cats)}
        windows_list = [base_label, recent_label]

        # Two rows: share (top) and absolute count (bottom)
        fig, axes = plt.subplots(2, TOP_N, figsize=(16, 8), sharey='row')
        fig.suptitle(f'{label} — KSI share and count base vs recent ({title_suffix})', fontsize=10)
        for col_idx, la_name in enumerate(priority_names):
            sub = grp[grp['la_name']==la_name]
            # Row 0: share
            ax_pct = axes[0, col_idx]
            bottoms_pct = [0] * len(windows_list)
            for cat in cats:
                vals = []
                for w in windows_list:
                    row = sub[(sub['window']==w) & (sub[col_name].astype(str)==cat)]
                    vals.append(row['pct'].values[0] if not row.empty else 0)
                ax_pct.bar(windows_list, vals, bottom=bottoms_pct,
                           color=cat_colors[cat], edgecolor='white', lw=0.5)
                bottoms_pct = [b+v for b,v in zip(bottoms_pct, vals)]
            ax_pct.set_title(la_name, fontsize=8, fontweight='bold')
            ax_pct.set_ylim(0, 105)
            ax_pct.tick_params(axis='x', labelsize=7, rotation=15)
            if col_idx == 0: ax_pct.set_ylabel('% of KSI')

            # Row 1: absolute count
            ax_cnt = axes[1, col_idx]
            bottoms_cnt = [0] * len(windows_list)
            for cat in cats:
                vals = []
                for w in windows_list:
                    row = sub[(sub['window']==w) & (sub[col_name].astype(str)==cat)]
                    vals.append(row['KSI'].values[0] if not row.empty else 0)
                ax_cnt.bar(windows_list, vals, bottom=bottoms_cnt,
                           color=cat_colors[cat], edgecolor='white', lw=0.5)
                bottoms_cnt = [b+v for b,v in zip(bottoms_cnt, vals)]
            ax_cnt.tick_params(axis='x', labelsize=7, rotation=15)
            ax_cnt.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
            if col_idx == 0: ax_cnt.set_ylabel('KSI count')

        handles = [plt.Rectangle((0,0),1,1,color=cat_colors[c]) for c in cats]
        fig.legend(handles, cats, loc='lower center', ncol=min(len(cats),7),
                   fontsize=7.5, bbox_to_anchor=(0.5,-0.04))
        plt.tight_layout()
        plt.show()
        savefig(f'fig_factor_{grp_name}_{col_name.replace("_label","").replace("_","")}')


# 7b. Dominant rising group per priority authority
# Identifies which road user group had the largest absolute KSI rise
# between base and recent periods for each worsening authority.

def dominant_group(df, la_names, base_yrs, recent_yrs):
    rows = []
    for la in la_names:
        sub = df[df['la_name']==la]
        base_counts   = (sub[sub['collision_year'].isin(base_yrs)]
                         .groupby('road_user').size()
                         .div(len(base_yrs)))
        recent_counts = (sub[sub['collision_year'].isin(recent_yrs)]
                         .groupby('road_user').size()
                         .div(len(recent_yrs)))
        rise = (recent_counts - base_counts).fillna(recent_counts).fillna(0)
        dom  = rise.idxmax() if len(rise) > 0 else 'Unknown'
        rows.append({
            'la_name'            : la,
            'dominant_group'     : dom,
            'avg_annual_rise'    : round(rise.max(), 1),
            'base_avg_total_ksi' : round(base_counts.sum(), 1),
            'recent_avg_total_ksi': round(recent_counts.sum(), 1),
        })
    return pd.DataFrame(rows)

dom_rising = dominant_group(ksi_complete, top5_w_names, BASE_YEARS, RECENT_YEARS)
print('=== Dominant rising road user group — top worsening authorities ===')
print(dom_rising.to_string(index=False))
dom_rising.to_csv(os.path.join(OUTPUT_DIR,'tab_dominant_rising_group.csv'), index=False)


# 7c. Priority authority summary table
# One row per priority authority covering all key metrics.
# Feeds directly into portfolio placeholder values.

def priority_summary(df, la_names, ols_df, base_yrs, recent_yrs):
    rows = []
    for la in la_names:
        sub     = df[df['la_name']==la]
        sub_b   = sub[sub['collision_year'].isin(base_yrs)]
        sub_r   = sub[sub['collision_year'].isin(recent_yrs)]
        ols_row = ols_df[ols_df['la_name']==la]

        base_avg   = len(sub_b) / len(base_yrs)
        recent_avg = len(sub_r) / len(recent_yrs)
        pct_change = (recent_avg - base_avg) / base_avg * 100 if base_avg > 0 else np.nan

        ru_recent  = sub_r['road_user'].value_counts()
        top_ru     = ru_recent.index[0] if len(ru_recent) > 0 else 'Unknown'
        top_ru_pct = ru_recent.iloc[0] / ru_recent.sum() * 100 if len(ru_recent) > 0 else np.nan

        spd_recent = sub_r['speed_limit'].dropna().value_counts()
        top_spd    = f'{int(spd_recent.index[0])} mph' if len(spd_recent) > 0 else 'Unknown'

        rt_recent  = sub_r['road_type_label'].dropna().value_counts()
        top_rt     = rt_recent.index[0] if len(rt_recent) > 0 else 'Unknown'

        ur_recent  = sub_r['ur_label'].dropna().value_counts(normalize=True)
        urban_pct  = round(ur_recent.get('Urban', 0) * 100, 1)
        rural_pct  = round(ur_recent.get('Rural', 0) * 100, 1)

        sub_r_h = sub_r.copy()
        sub_r_h['hour'] = pd.to_datetime(sub_r_h['time'], format='%H:%M', errors='coerce').dt.hour
        hour_counts = sub_r_h['hour'].dropna().value_counts()
        peak_hr     = f'{int(hour_counts.index[0]):02d}:00' if len(hour_counts) > 0 else 'Unknown'

        row = {
            'la_name'          : la,
            'base_avg_ksi'     : round(base_avg, 1),
            'recent_avg_ksi'   : round(recent_avg, 1),
            'pct_change'       : round(pct_change, 1),
            'ols_slope'        : round(ols_row['slope'].values[0], 2) if len(ols_row) > 0 else np.nan,
            'ci_lower'         : round(ols_row['ci_lower'].values[0], 2) if len(ols_row) > 0 else np.nan,
            'ci_upper'         : round(ols_row['ci_upper'].values[0], 2) if len(ols_row) > 0 else np.nan,
            'r_squared'        : round(ols_row['r_squared'].values[0], 3) if len(ols_row) > 0 else np.nan,
            'ci_includes_zero' : ols_row['ci_includes_zero'].values[0] if len(ols_row) > 0 else np.nan,
            'dominant_road_user': top_ru,
            'dominant_ru_pct'  : round(top_ru_pct, 1),
            'top_speed_zone'   : top_spd,
            'top_road_type'    : top_rt,
            'urban_pct'        : urban_pct,
            'rural_pct'        : rural_pct,
            'peak_hour'        : peak_hr,
        }
        rows.append(row)
    return pd.DataFrame(rows)

summary_w = priority_summary(ksi_complete, top5_w_names, ols, BASE_YEARS, RECENT_YEARS)
summary_i = priority_summary(ksi_complete, top5_i_names, ols, BASE_YEARS, RECENT_YEARS)
priority_summary_df = pd.concat([
    summary_w.assign(group='worsening'),
    summary_i.assign(group='improving')
], ignore_index=True)

print('=== Priority authority summary ===')
print(priority_summary_df.to_string(index=False))
priority_summary_df.to_csv(os.path.join(OUTPUT_DIR,'tab_priority_authority_summary.csv'), index=False)


# ── 8. Robustness and Sensitivity Checks ──────────────────────────────────────

# 8.1 Raw vs adjusted KSI trend direction — COVID excluded
non_covid = adj[~adj['collision_year'].isin(COVID_YEARS)]
print('=== Robustness Check 1: Trend direction — raw vs adjusted KSI (COVID excluded) ===')
for label, col_name in [('Raw KSI','raw_ksi'),('Adjusted KSI','adj_ksi')]:
    x    = non_covid['collision_year'].values
    y    = non_covid[col_name].values
    slope, intercept, r, p, se = stats.linregress(x, y)
    t_crit   = stats.t.ppf(0.975, df=len(x)-2)
    ci_lower = slope - t_crit * se
    ci_upper = slope + t_crit * se
    direction = 'UPWARD' if slope>0 else 'DOWNWARD'
    print(f'{label}: slope = {slope:+.1f}/yr  95% CI [{ci_lower:.1f}, {ci_upper:.1f}]  ({direction})')
print('Note: adjusted KSI trend unreliable — partial IBRS adoption inflates earlier years.')

fig, axes = plt.subplots(1,2,figsize=(14,5))
for ax, col_name, label, color in [
    (axes[0],'raw_ksi','Raw KSI','#e67e22'),
    (axes[1],'adj_ksi','Adjusted KSI (partial)','#2ecc71')
]:
    ax.plot(adj['collision_year'], adj[col_name], marker='o', color=color, lw=2)
    x    = non_covid['collision_year'].values
    y    = non_covid[col_name].values
    coef = np.polyfit(x - x.mean(), y, 1)
    y_fit = coef[0]*(adj['collision_year'].values-x.mean()) + coef[1]
    ax.plot(adj['collision_year'], y_fit, linestyle=':', color='#2c3e50', lw=1.5,
            label=f'Trend (slope {coef[0]:+.0f}/yr)')
    for yr in COVID_YEARS:
        ax.axvspan(yr-0.4,yr+0.4, alpha=0.1, color='grey')
    ax.set_title(label, fontweight='bold')
    ax.set_ylabel('Collisions')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
    ax.legend(fontsize=8)
plt.suptitle('Fig 12: Robustness check 1 — trend direction raw vs adjusted (COVID excluded)',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show(); savefig('fig_12_robustness_trend')

# 8.2 Hotspot definition sensitivity — computed in Section 5

# 8.3 OLS excluding COVID years
print('=== Robustness Check 3: OLS excluding COVID years per LA ===')
ols_excl_rows = []
for la_name, grp in annual.groupby('la_name'):
    grp_excl = grp[~grp['collision_year'].isin(COVID_YEARS)].dropna()
    if len(grp_excl) < MIN_YEARS: continue
    x = grp_excl['collision_year'].values
    y = grp_excl['KSI'].values
    slope, _, _, _, _ = stats.linregress(x, y)
    ols_excl_rows.append({'la_name': la_name, 'slope_excl_covid': slope})
ols_excl = pd.DataFrame(ols_excl_rows)
ols_comp = ols.merge(ols_excl, on='la_name', how='left')
ols_comp['direction_changes'] = (ols_comp['slope']>0) != (ols_comp['slope_excl_covid']>0)
n_change = ols_comp['direction_changes'].sum()
print(f'LAs where direction changes when COVID excluded: {n_change} / {len(ols_comp)}')
if n_change > 0:
    print(ols_comp[ols_comp['direction_changes']][
        ['la_name','slope','slope_excl_covid']].round(2).to_string(index=False))
else:
    print('No LA changes direction — trend conclusions robust to COVID exclusion.')

# 8.4 Minimum KSI threshold sensitivity
print('=== Robustness Check 4: MIN_KSI_BASE sensitivity ===')
for threshold in [20, 30, 50]:
    ols_t = pd.DataFrame(ols_rows)
    ols_t = ols_t[ols_t['mean_ksi'] >= threshold]
    w = ols_t[ols_t['slope']>0].sort_values('slope',ascending=False).head(TOP_N)['la_name'].tolist()
    i = ols_t[ols_t['slope']<0].sort_values('slope').head(TOP_N)['la_name'].tolist()
    print(f'\nThreshold = {threshold} KSI/yr ({len(ols_t)} LAs):')
    print(f'  Top {TOP_N} worsening : {w}')
    print(f'  Top {TOP_N} improving : {i}')

# 8.5 Composite weighting sensitivity — OLS slope ranking
# Tests whether the priority authority ranking is stable across different
# weightings of absolute rise vs percentage change.
print('=== Robustness Check 5: Composite weighting sensitivity ===')

worsening_all = ols[ols['slope']>0].copy()
worsening_all['absolute_rise'] = worsening_all['recent_avg'] - worsening_all['base_avg']
abs_min, abs_max = worsening_all['absolute_rise'].min(), worsening_all['absolute_rise'].max()
pct_min, pct_max = worsening_all['avg_pct_change'].min(), worsening_all['avg_pct_change'].max()
worsening_all['norm_abs'] = (worsening_all['absolute_rise'] - abs_min) / (abs_max - abs_min + 1e-9)
worsening_all['norm_pct'] = (worsening_all['avg_pct_change'] - pct_min) / (pct_max - pct_min + 1e-9)

schemes = {'50/50': (0.5, 0.5), '60/40': (0.6, 0.4), '70/30': (0.7, 0.3)}
scheme_ranks = {}
for scheme_name, (w_abs, w_pct) in schemes.items():
    worsening_all[f'composite_{scheme_name}'] = (
        w_abs * worsening_all['norm_abs'] + w_pct * worsening_all['norm_pct']
    )
    scheme_ranks[scheme_name] = (
        worsening_all.sort_values(f'composite_{scheme_name}', ascending=False)
        .reset_index(drop=True)
        .assign(**{f'rank_{scheme_name}': lambda df: df.index + 1})
        [['la_name', f'rank_{scheme_name}']]
    )

rank_compare = scheme_ranks['50/50']
for s in ['60/40', '70/30']:
    rank_compare = rank_compare.merge(scheme_ranks[s], on='la_name')

rank_compare = rank_compare[rank_compare['rank_60/40'] <= 10].sort_values('rank_60/40')
rank_compare['max_rank_shift'] = rank_compare[
    ['rank_50/50', 'rank_60/40', 'rank_70/30']
].apply(lambda r: r.max() - r.min(), axis=1)
rank_compare['sensitive'] = rank_compare['max_rank_shift'] >= 3

print(rank_compare[['la_name','rank_50/50','rank_60/40','rank_70/30',
                     'max_rank_shift','sensitive']].to_string(index=False))
print(f'\nPrimary scheme: 60/40 (absolute rise weighted higher, reflecting client priority).')
sensitive_las = rank_compare[rank_compare['sensitive']]['la_name'].tolist()
if sensitive_las:
    print(f'Authorities sensitive to weighting (rank shifts 3+): {sensitive_las}')
    print('These authorities are borderline — treat ranking position with additional caution.')
else:
    print('Top authority rankings are stable across all three weighting schemes.')
rank_compare.to_csv(os.path.join(OUTPUT_DIR,'tab_composite_weighting_sensitivity.csv'), index=False)


# ── 9. OLS Diagnostics and Validation ─────────────────────────────────────────

fig, axes = plt.subplots(1, TOP_N, figsize=(16,4))
fig.suptitle(f'Fig 13: OLS residuals — top {TOP_N} worsening LAs',
             fontweight='bold', fontsize=10)
for ax, la_name in zip(axes, top5_w_names):
    sub = annual[annual['la_name']==la_name].dropna()
    row = ols[ols['la_name']==la_name].iloc[0]
    fitted    = row['intercept'] + row['slope'] * sub['collision_year']
    residuals = sub['KSI'].values - fitted.values
    covid_mask = sub['collision_year'].isin(COVID_YEARS)
    ax.scatter(sub['collision_year'][~covid_mask], residuals[~covid_mask],
               color='#3498db', s=40, zorder=3, label='Non-COVID')
    ax.scatter(sub['collision_year'][covid_mask], residuals[covid_mask],
               color='#e74c3c', s=60, marker='D', zorder=4, label='COVID')
    ax.axhline(0, color='black', lw=0.8, linestyle='--')
    ax.set_title(la_name, fontsize=8, fontweight='bold')
    ax.set_xlabel('Year'); ax.tick_params(axis='x', rotation=45)
    if ax == axes[0]:
        ax.set_ylabel('Residual'); ax.legend(fontsize=7)
plt.tight_layout(); plt.show(); savefig('fig_13_ols_residuals')
print('Red diamonds = COVID years.')

print('=== COVID influence on OLS slope ===')
all_priority = top5_w_names + top5_i_names
rows = []
for la_name in all_priority:
    sub      = annual[annual['la_name']==la_name].dropna()
    sub_excl = sub[~sub['collision_year'].isin(COVID_YEARS)]
    if len(sub_excl) < 4: continue
    slope_full, _, _, _, _ = stats.linregress(sub['collision_year'], sub['KSI'])
    slope_excl, _, _, _, _ = stats.linregress(sub_excl['collision_year'], sub_excl['KSI'])
    rows.append({'LA': la_name,
                 'Slope (full)': round(slope_full,2),
                 'Slope (no COVID)': round(slope_excl,2),
                 'Difference': round(slope_excl-slope_full,2),
                 'Direction same': (slope_full>0)==(slope_excl>0)})
influence_df = pd.DataFrame(rows)
print(influence_df.to_string(index=False))
influence_df.to_csv(os.path.join(OUTPUT_DIR,'tab_covid_influence.csv'), index=False)

# 9b. Bootstrap CI validation — top worsening authorities
# Resamples annual KSI observations per authority and compares bootstrap 95% CI
# to the analytical OLS CI. Close agreement validates the analytical CI assumptions.
# Material divergence flags that authority's CI as unreliable.

print(f'=== Bootstrap CI validation — top {TOP_N} worsening LAs ({N_BOOTSTRAP} resamples) ===')
np.random.seed(42)
boot_rows = []
for la_name in top5_w_names:
    sub = annual[annual['la_name']==la_name].dropna()
    x   = sub['collision_year'].values
    y   = sub['KSI'].values
    xc  = x - x.mean()

    # Analytical CI
    slope_ols, _, _, _, se = stats.linregress(xc, y)
    t_crit   = stats.t.ppf(0.975, df=len(x)-2)
    ci_lo_ols = slope_ols - t_crit * se
    ci_hi_ols = slope_ols + t_crit * se

    # Bootstrap CI
    boot_slopes = []
    idx = np.arange(len(x))
    for _ in range(N_BOOTSTRAP):
        bidx = np.random.choice(idx, size=len(idx), replace=True)
        bx = xc[bidx]; by = y[bidx]
        if np.std(bx) > 0:
            bs, _, _, _, _ = stats.linregress(bx, by)
            boot_slopes.append(bs)
    ci_lo_boot = np.percentile(boot_slopes, 2.5)
    ci_hi_boot = np.percentile(boot_slopes, 97.5)

    # Flag if CIs diverge materially (bootstrap width > 1.5x analytical width)
    analytical_width = ci_hi_ols - ci_lo_ols
    bootstrap_width  = ci_hi_boot - ci_lo_boot
    diverges = bootstrap_width > 1.5 * analytical_width

    boot_rows.append({
        'la_name'          : la_name,
        'ols_slope'        : round(slope_ols, 2),
        'analytical_ci_lo' : round(ci_lo_ols, 2),
        'analytical_ci_hi' : round(ci_hi_ols, 2),
        'bootstrap_ci_lo'  : round(ci_lo_boot, 2),
        'bootstrap_ci_hi'  : round(ci_hi_boot, 2),
        'ci_diverges'      : diverges,
    })

boot_df = pd.DataFrame(boot_rows)
print(boot_df.to_string(index=False))
diverging = boot_df[boot_df['ci_diverges']]['la_name'].tolist()
if diverging:
    print(f'\nAuthorities where bootstrap CI diverges from analytical CI: {diverging}')
    print('Treat analytical CI for these authorities with additional caution.')
else:
    print('\nBootstrap and analytical CIs are consistent — OLS CI assumptions are reasonable.')
boot_df.to_csv(os.path.join(OUTPUT_DIR,'tab_bootstrap_ci_comparison.csv'), index=False)


# ── 9c. Poisson / Negative Binomial Regression ────────────────────────────────
# Overdispersion tested automatically; NB used if Pearson chi2/df > 1.5.

ksi_confirmed = full[
    full['ksi'] & (~full['provisional']) &
    full['la_name'].notna() &
    full['collision_year'].between(YEAR_START, YEAR_END)
].copy()

ksi_confirmed['speed_cat'] = pd.cut(
    ksi_confirmed['speed_limit'],
    bins=[0,20,30,40,100],
    labels=['20 mph or below','30 mph','40 mph','50 mph or above'],
    right=True
).astype(str).replace('nan', np.nan)

ksi_confirmed['road_user_group'] = ksi_confirmed['casualty_type'].map(ROAD_USER_MAP).fillna('Other')
ksi_confirmed['ur_label']        = ksi_confirmed['urban_or_rural_area'].map(URBAN_MAP)

model_agg = (ksi_confirmed
             .groupby(['la_name','collision_year','road_user_group',
                       'speed_cat','road_type_label','ur_label'])
             .size().reset_index(name='ksi_count'))

if os.path.exists(POP_FILE):
    ons_pop_m = pd.read_csv(POP_FILE, skiprows=POP_SKIPROWS)[[POP_CODE_COL, POP_VAL_COL]].rename(
        columns={POP_CODE_COL:'local_authority_ons_district', POP_VAL_COL:'population'})
    ons_pop_m['population'] = pd.to_numeric(ons_pop_m['population'], errors='coerce')
    ons_pop_m = ons_pop_m[ons_pop_m['local_authority_ons_district'].notna()]
    la_code_m = col[['la_name','local_authority_ons_district']].drop_duplicates().dropna()
    pop_m     = la_code_m.merge(ons_pop_m, on='local_authority_ons_district', how='left')
    model_agg = model_agg.merge(pop_m[['la_name','population']], on='la_name', how='left')
    offset_col  = 'population'
    offset_note = 'log(population)'
else:
    col_counts = (col_window.groupby(['la_name','collision_year'])
                  ['collision_index'].nunique().reset_index(name='n_collisions'))
    model_agg   = model_agg.merge(col_counts, on=['la_name','collision_year'], how='left')
    offset_col  = 'n_collisions'
    offset_note = 'log(n_collisions)'
    print('Population file not found — using collision count as offset.')

model_agg = model_agg.dropna(
    subset=['ksi_count',offset_col,'road_user_group','speed_cat','road_type_label','ur_label']
).copy()
model_agg[offset_col] = model_agg[offset_col].clip(lower=1)

model_agg['road_user_group'] = pd.Categorical(model_agg['road_user_group'], categories=ROAD_USER_ORDER)
model_agg['speed_cat']       = pd.Categorical(
    model_agg['speed_cat'],
    categories=['30 mph','20 mph or below','40 mph','50 mph or above'])

formula = (f'ksi_count ~ road_user_group + speed_cat + road_type_label + ur_label'
           f' + C(collision_year) + offset(np.log({offset_col}))')

model_poisson    = smf.glm(formula=formula, data=model_agg, family=sm.families.Poisson()).fit(disp=False)
dispersion_ratio = model_poisson.pearson_chi2 / model_poisson.df_resid
print(f'\nPoisson dispersion ratio: {dispersion_ratio:.2f}  (>1.5 = overdispersed)')

if dispersion_ratio > 1.5:
    model_nb         = smf.glm(formula=formula, data=model_agg, family=sm.families.NegativeBinomial()).fit(disp=False)
    final_model      = model_nb
    final_model_name = 'Negative Binomial'
else:
    final_model      = model_poisson
    final_model_name = 'Poisson'

print(f'Final model: {final_model_name}  |  Poisson AIC: {model_poisson.aic:.1f}', end='')
if dispersion_ratio > 1.5: print(f'  NB AIC: {model_nb.aic:.1f}')
else: print()

results = pd.DataFrame({
    'term'    : final_model.params.index,
    'RR'      : np.exp(final_model.params),
    'CI_lower': np.exp(final_model.conf_int()[0]),
    'CI_upper': np.exp(final_model.conf_int()[1]),
    'p_value' : final_model.pvalues
}).reset_index(drop=True)

results_clean = results[
    ~results['term'].str.startswith('C(collision_year)') &
    (results['term'] != 'Intercept')
].copy()
results_clean['significant'] = results_clean['p_value'] < 0.05
results_clean = results_clean.sort_values('RR')

print(f'\n=== {final_model_name} rate ratios ===')
print(results_clean[['term','RR','CI_lower','CI_upper','p_value','significant']].round(3).to_string(index=False))
results_clean.to_csv(
    os.path.join(OUTPUT_DIR, f'tab_model_results_{final_model_name.lower().replace(" ","_")}.csv'),
    index=False)

fig, ax = plt.subplots(figsize=(9, max(5, len(results_clean)*0.4)))
colors_rr = ['#e74c3c' if s else '#95a5a6' for s in results_clean['significant']]
y_pos = range(len(results_clean))
ax.barh(list(y_pos), results_clean['RR']-1, left=1, color=colors_rr, edgecolor='white', height=0.6)
ax.errorbar(x=results_clean['RR'], y=list(y_pos),
            xerr=[results_clean['RR']-results_clean['CI_lower'],
                  results_clean['CI_upper']-results_clean['RR']],
            fmt='none', color='#2c3e50', capsize=3, linewidth=1)
ax.axvline(1, color='black', linewidth=0.8, linestyle='--')
ax.set_yticks(list(y_pos)); ax.set_yticklabels(results_clean['term'], fontsize=8)
ax.set_xscale('log')
ax.set_xlabel('Rate ratio (log scale)')
ax.set_title(f'Fig 14: {final_model_name} rate ratios — KSI risk factors\n'
             f'Offset: {offset_note}. Red = p < 0.05.', fontweight='bold')
plt.tight_layout(); plt.show()
savefig(f'fig_14_model_rate_ratios_{final_model_name.lower().replace(" ","_")}')

year_coefs = results[results['term'].str.startswith('C(collision_year)')].copy()
year_coefs['year'] = year_coefs['term'].str.extract(r'(\d{4})').astype(int)
year_coefs = year_coefs.sort_values('year')
print('\n=== Direction agreement: OLS vs Poisson/NB year coefficients ===')
print(f'OLS top {TOP_N} worsening: {top5_w_names}')
print(f'OLS top {TOP_N} improving: {top5_i_names}')
print(year_coefs[['year','RR','CI_lower','CI_upper','p_value']].round(3).to_string(index=False))


# ── 10. Policy Evaluation ─────────────────────────────────────────────────────
# Speed limit distribution and overall KSI trend for POLICY_LAS.
# Update POLICY_LAS in config to change which authorities are evaluated.

policy_data = ksi_complete[ksi_complete['la_name'].isin(POLICY_LAS)].copy()

fig, axes = plt.subplots(1, len(POLICY_LAS), figsize=(7*len(POLICY_LAS), 5))
if len(POLICY_LAS) == 1: axes = [axes]
for ax, la_name in zip(axes, POLICY_LAS):
    sub = (policy_data[policy_data['la_name']==la_name]
           .groupby(['collision_year','speed_limit']).size().unstack(fill_value=0))
    for sl, color in [(20,'#e74c3c'),(30,'#3498db')]:
        if sl in sub.columns:
            ax.plot(sub.index, sub[sl], marker='o', lw=2, color=color, label=f'{sl} mph')
    ax.axvspan(min(COVID_YEARS)-0.5, max(COVID_YEARS)+0.5, alpha=0.08, color='grey')
    ax.set_title(f'{la_name} — KSI by speed limit over time', fontweight='bold')
    ax.set_ylabel('KSI casualties'); ax.set_xlabel('Year'); ax.legend()
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
plt.suptitle(f'Fig 15: 20mph vs 30mph KSI trend — {", ".join(POLICY_LAS)}',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show(); savefig('fig_15_20mph_policy')

fig, ax = plt.subplots(figsize=(12,5))
palette_p = plt.cm.get_cmap('tab10', len(POLICY_LAS))
for i, la_name in enumerate(POLICY_LAS):
    sub = policy_data[policy_data['la_name']==la_name].groupby('collision_year').size()
    row = ols[ols['la_name']==la_name]
    if len(row) == 0:
        ax.plot(sub.index, sub.values, marker='o', lw=2, color=palette_p(i), label=la_name)
    else:
        ax.plot(sub.index, sub.values, marker='o', lw=2, color=palette_p(i),
                label=f"{la_name} (slope={row['slope'].values[0]:+.1f}/yr, "
                      f"R²={row['r_squared'].values[0]:.2f})")
ax.axvspan(min(COVID_YEARS)-0.5, max(COVID_YEARS)+0.5, alpha=0.08, color='grey', label='COVID period')
ax.set_title(f'{" vs ".join(POLICY_LAS)} — overall KSI trend ({YEAR_START}–{YEAR_END})',
             fontweight='bold')
ax.set_ylabel('KSI casualties'); ax.set_xlabel('Year')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.legend(fontsize=9)
plt.suptitle(f'Fig 16: {" vs ".join(POLICY_LAS)} — KSI trend comparison',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show(); savefig('fig_16_policy_comparison')

print('\nKey comparison:')
for la_name in POLICY_LAS:
    row = ols[ols['la_name']==la_name]
    if len(row) == 0:
        print(f'{la_name}: not in OLS results (insufficient data)')
    else:
        row = row.iloc[0]
        print(f'{la_name}: slope={row["slope"]:+.1f}/yr  R²={row["r_squared"]:.2f}  '
              f'CI [{row["ci_lower"]:.1f}, {row["ci_upper"]:.1f}]')


# ── 11. Material Change Threshold ─────────────────────────────────────────────
# Flag LAs where recent average exceeds base average by >= THRESHOLD_PCT
# AND OLS slope is positive.
# Red = CI excludes zero (statistically clear). Orange = CI includes zero.

print(f'=== Material Change Threshold: >= {THRESHOLD_PCT}% above baseline AND positive slope ===')

flagged = ols[
    (ols['avg_pct_change'] > THRESHOLD_PCT) & (ols['slope'] > 0)
].sort_values('avg_pct_change', ascending=False)

print(f'{len(flagged)} LAs flagged for intervention review:')
cols_show = ['la_name','slope','r_squared','base_avg','recent_avg','avg_pct_change','ci_includes_zero']
print(flagged[cols_show].head(10).round(2).to_string(index=False))
flagged[cols_show].to_csv(os.path.join(OUTPUT_DIR,'tab_material_change_flagged.csv'), index=False)

fig, ax = plt.subplots(figsize=(12,6))
top_flagged = flagged.head(15).sort_values('avg_pct_change')
colors_m = ['#e74c3c' if not ci else '#f39c12' for ci in top_flagged['ci_includes_zero']]
ax.barh(top_flagged['la_name'], top_flagged['avg_pct_change'], color=colors_m, edgecolor='white')
ax.axvline(THRESHOLD_PCT, color='black', lw=1, linestyle='--',
           label=f'{THRESHOLD_PCT}% trigger threshold')
ax.set_xlabel(f'% change vs {base_label} baseline')
ax.set_title('Top 15 LAs exceeding material change threshold', fontweight='bold')
ax.legend(fontsize=9)
plt.suptitle('Fig 17: Material change flags', fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show(); savefig('fig_17_material_change')
print('Red = CI excludes zero (statistically clear). Orange = CI includes zero (treat with caution).')